# Xavier's Assistant: full website crawl on Google Colab

This notebook builds the chatbot's knowledge base (the full deep crawl of sxca.edu.in plus bge-m3 embeddings) on Google's free cloud machine. Afterwards you download one zip file and unpack it on your laptop.

**Why:** embedding thousands of pages and PDFs takes hours on a laptop CPU but only minutes on Colab's GPU. OCR (Tesseract with Hindi and Gujarati) also works here, so scanned notices get read.

**Before you start:** *Runtime → Change runtime type → **T4 GPU***.

| Stage | What it does | Time |
|---|---|---|
| 1. Crawl | Discover and download pages, PDFs and notices, then extract and clean the text | ~1.5–2 h (polite 1 s delay between requests) |
| 2. Index | Chunk and embed on the GPU, store in ChromaDB | ~10–20 min |

A checkpoint is saved to your Google Drive after each stage (and whenever you run the checkpoint cell). If Colab disconnects, reconnect and **run all cells again**: the crawl resumes, skipping everything already done.

## 1. Check the GPU and connect Google Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU (stage 1 works without it, stage 2 is much slower)"

from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/25pai028-art/Xaviersbot.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/xaviersbot"   # checkpoints and the final zip go here
DATA_DIR = "/content/data"                        # working copy (fast local disk)
CHROMADB_VERSION = "1.5.9"                        # must match the laptop's chromadb version

import os
os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. Get the code

The repository is private, so paste a **GitHub token** with read access. To create one: github.com, then Settings → Developer settings → Fine-grained tokens → *Only select repositories: Xaviersbot* → *Contents: Read-only*.

The token is typed into a hidden prompt, used once, and not saved anywhere.

In [ ]:
import getpass, subprocess, shutil
if os.path.exists("/content/Xaviersbot"):
    shutil.rmtree("/content/Xaviersbot")
token = getpass.getpass("GitHub token (hidden): ").strip()
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/content/Xaviersbot"],
                   capture_output=True, text=True)
del token, url
print("cloned" if r.returncode == 0 else r.stderr.replace(REPO, "<repo>"))
%cd /content/Xaviersbot
!git log --oneline -1

## 3. Install dependencies (about 3–5 minutes)

In [ ]:
# OCR engine with English, Hindi and Gujarati
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-guj > /dev/null && tesseract --list-langs | tail -n +2 | tr '\n' ' '

# Python packages. Colab's own GPU build of PyTorch is kept (the requirements file pins the CPU build for the laptop).
!grep -vE '^(torch|--extra-index-url)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt "chromadb=={CHROMADB_VERSION}"

# Headless browser for JavaScript-rendered pages
!playwright install --with-deps chromium > /dev/null 2>&1 && echo "chromium ready"

## 4. Settings for this machine

In [ ]:
settings = f"""
DATA_DIR={DATA_DIR}
EMBEDDING_DEVICE=cuda
EMBEDDING_BATCH_SIZE=32
TESSERACT_CMD=/usr/bin/tesseract
CRAWL_USE_PLAYWRIGHT=true
"""
open(".env", "w").write(settings)
print(settings)

## 5. Checkpoints on Google Drive

In [ ]:
import time, zipfile

CHECKPOINT = f"{DRIVE_DIR}/xaviersbot-data.zip"

def save_checkpoint():
    """Zip the knowledge base (SQLite + ChromaDB) to Google Drive."""
    tmp = "/content/xaviersbot-data.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(DATA_DIR):
            if "/cache" in root:
                continue  # model caches are re-downloaded anyway
            for f in files:
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, DATA_DIR))
    shutil.copy(tmp, CHECKPOINT)
    print(f"checkpoint saved: {CHECKPOINT} ({os.path.getsize(CHECKPOINT) / 1e6:.0f} MB) at {time.strftime('%H:%M')}")

def run(module, *args):
    # Run a project script (e.g. scripts.crawl) with command-line arguments inside this notebook.
    # (IPython's %run would reject options like --no-index.)
    import importlib, sys
    if "/content/Xaviersbot" not in sys.path:
        sys.path.insert(0, "/content/Xaviersbot")
    sys.argv = [module, *args]
    try:
        importlib.import_module(module).main()
    except SystemExit as e:
        if e.code not in (0, None):
            raise

# Resume from an earlier checkpoint if there is one
if os.path.exists(CHECKPOINT) and not os.path.exists(f"{DATA_DIR}/sxca.db"):
    os.makedirs(DATA_DIR, exist_ok=True)
    zipfile.ZipFile(CHECKPOINT).extractall(DATA_DIR)
    print("restored checkpoint from Drive, the crawl will resume")
else:
    print("starting fresh" if not os.path.exists(f"{DATA_DIR}/sxca.db") else "using the existing working copy")

## 6. Stage 1: crawl and clean

This downloads the whole website (pages, PDFs, DOCX, image notices) and extracts clean text. It doesn't embed anything yet.

- You can stop it anytime with *Runtime → Interrupt execution*. Then run the checkpoint cell below.
- To test first, use the `--max-pages 50` variant shown in the cell comment.

In [ ]:
try:
    run("scripts.crawl", "--no-index")   # test first with: run("scripts.crawl", "--no-index", "--max-pages", "50")
finally:
    save_checkpoint()

## 7. Stage 2: embed everything on the GPU

In [ ]:
try:
    run("scripts.crawl", "--index-only")
finally:
    save_checkpoint()

## 8. Check the result

In [ ]:
run("scripts.crawl", "--stats")

In [ ]:
# Quick retrieval check with the evaluation questions (no LLM needed)
run("scripts.evaluate", "--file", "tests/eval/student_questions.jsonl")

## 9. Download the knowledge base

The zip is already on your Google Drive (`MyDrive/xaviersbot/xaviersbot-data.zip`). This cell also downloads it directly.

**Install it on the laptop** (in cmd, inside the `XaviersBot` folder):
1. Stop the chat server (Ctrl+C).
2. Keep the old data just in case: `ren data data-old`
3. Unzip: `powershell Expand-Archive xaviersbot-data.zip -DestinationPath data`
4. Check: `python -m scripts.crawl --stats`
5. Start the chat: `uvicorn app.main:app`

Later updates can run incrementally on the laptop with `python -m scripts.crawl`, which only processes what changed on the website, or with this notebook again.

In [ ]:
from google.colab import files
files.download(CHECKPOINT)